# Stage B.2 — Fit speaker-type judgments (vigilant model)

Reproduces the **speaker-type column of Table 1** in the paper. The vigilant
listener model predicts a marginal posterior over the speaker's goal
ψ ∈ {pers-, inf, pers+} after each utterance. We fit `vigilant_F`
(`update_internal=False`) to participants' categorical Skeptic / Scientist /
Promoter judgments and compute the mean Jensen-Shannon distance between
predicted and empirical proportions.

Two priors are reported:

| column | prior on ψ |
|---|---|
| **Uniform** | `[1/3, 1/3, 1/3]` (no preference) |
| **Truth-Default** | `[0.25, 0.50, 0.25]` (informative speaker more likely a priori) |

For the Truth-Default column, we sweep over candidate priors of the form
`[(1−p)/2, p, (1−p)/2]` (symmetric, parameterized by P(inf)) and report
results at the prior closest to the paper's reported `[0.25, 0.50, 0.25]`.

Headline finding: under either prior, the best-fitting α for **all six
persuasive sequences** floors out near 0.10 — even after hearing
"some successful" 4 of 5 rounds, participants don't infer the speaker is a
Promoter. The truth-default prior reduces mean JS from 0.206 → 0.171, but
the persuasive-sequence α values stay near floor.

## Setup

In [15]:
import time
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from rsa_optimal_exp_core import World, PragmaticListener_obs_n


THETA_VALUES = np.linspace(0, 1, 11)
ALPHA_GRID = np.logspace(np.log10(0.1), np.log10(100), 200)   # 200 log-spaced α
PSI_MAP = {"anti": 0, "neutral": 1, "pro": 2}                  # data → model ψ index

UTTERANCE_SEQUENCES = {
    ("informative", 0): ["most,successful", "no,unsuccessful", "all,successful", "no,unsuccessful", "all,successful"],
    ("informative", 1): ["most,unsuccessful", "all,unsuccessful", "no,successful", "all,unsuccessful", "no,successful"],
    ("pers_plus", 0): ["some,successful", "some,successful", "some,unsuccessful", "some,successful", "some,successful"],
    ("pers_plus", 1): ["most,successful", "some,successful", "some,unsuccessful", "some,successful", "some,unsuccessful"],
    ("pers_plus", 2): ["most,successful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,successful"],
    ("pers_minus", 0): ["some,unsuccessful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,unsuccessful"],
    ("pers_minus", 1): ["most,unsuccessful", "some,unsuccessful", "some,successful", "some,unsuccessful", "some,successful"],
    ("pers_minus", 2): ["most,unsuccessful", "some,successful", "some,unsuccessful", "some,successful", "some,unsuccessful"],
}


## Load empirical speaker-type proportions

`speaker_type.csv` is produced by `process_data.ipynb` from the vigilant-
condition participants' per-round speaker-type responses.

In [16]:
df_spk = pd.read_csv("./speaker_type.csv")
df_spk["speaker_cond"] = df_spk["speaker_sequence"].str.rsplit("_", n=1).str[0]
df_spk["seq_idx"] = df_spk["speaker_sequence"].str.rsplit("_", n=1).str[1].astype(int)

# Build {(speaker_cond, seq_idx): np.array shape (5, 3)} of empirical P(ψ | round)
empirical_psi = {}
for (sc, si), grp in df_spk.groupby(["speaker_cond", "seq_idx"]):
    dist = np.zeros((5, 3))
    for _, row in grp.iterrows():
        r = int(row["round"]) - 1
        dist[r, PSI_MAP[row["speaker_type"]]] = row["percentage"] / 100.0
    for r in range(5):
        if dist[r].sum() > 0:
            dist[r] /= dist[r].sum()
    empirical_psi[(sc, si)] = dist
print(f"Loaded empirical ψ distributions for {len(empirical_psi)} sequences")


Loaded empirical ψ distributions for 8 sequences


## Compute model predictions over (α × prior × sequence)

For the Uniform-Prior fit we only need the listener's default prior; for the
Truth-Default fit we sweep the prior. Doing one big nested loop here so
both columns of the table can be derived from a single computation.

In [17]:
# Sweep over candidate priors for the prior-search column. Symmetric form:
# prior = [(1−P_inf)/2, P_inf, (1−P_inf)/2].
P_INF_GRID = np.linspace(0, 1, 11)

world = World(n=1, m=5, theta_values=THETA_VALUES)

# Structure: predictions[p_inf][seq][alpha] = (5, 3) posterior-ψ array.
# Note: P_inf=1/3 ≈ 0.333 is the "uniform" case — included via separate evaluation below.
predictions = {}
t0 = time.time()
print(f"Building predictions for {len(P_INF_GRID)} priors × {len(UTTERANCE_SEQUENCES)} seqs × {len(ALPHA_GRID)} α values...")
for p_idx, p_inf in enumerate(P_INF_GRID):
    p_other = (1 - p_inf) / 2.0
    prior = np.array([p_other, p_inf, p_other])
    predictions[p_inf] = {}
    for seq, utterances in UTTERANCE_SEQUENCES.items():
        predictions[p_inf][seq] = {}
        for alpha in ALPHA_GRID:
            listener = PragmaticListener_obs_n(
                world=world, level=1, omega="strat",
                update_internal=False, alpha=alpha, beta=0.0,
                initial_beliefs_psi=prior,
            )
            preds = []
            for u in utterances:
                listener.listen_and_update(u)
                preds.append(listener.current_belief_psi.copy())
            predictions[p_inf][seq][alpha] = np.asarray(preds)
    if p_idx % 5 == 0 or p_idx == len(P_INF_GRID) - 1:
        print(f"  P_inf={p_inf:.3f}  ({time.time()-t0:.0f}s elapsed)")

# Also build predictions under the exact uniform prior [1/3, 1/3, 1/3].
uniform_prior = np.array([1/3, 1/3, 1/3])
preds_uniform = {}
for seq, utterances in UTTERANCE_SEQUENCES.items():
    preds_uniform[seq] = {}
    for alpha in ALPHA_GRID:
        listener = PragmaticListener_obs_n(
            world=world, level=1, omega="strat",
            update_internal=False, alpha=alpha, beta=0.0,
            initial_beliefs_psi=uniform_prior,
        )
        preds = []
        for u in utterances:
            listener.listen_and_update(u)
            preds.append(listener.current_belief_psi.copy())
        preds_uniform[seq][alpha] = np.asarray(preds)
print(f"All predictions ready in {time.time()-t0:.0f}s")


Building predictions for 11 priors × 8 seqs × 200 α values...


/Users/kangke/projects/prag_net/data/cogsci_rsa_listener_experiment_n5_o1/rsa_optimal_exp_core.py:382: UserWarning: theta values above precision is rounded to 10 decimals
  warnings.warn("theta values above precision is rounded to 10 decimals",


  P_inf=0.000  (19s elapsed)
  P_inf=0.500  (119s elapsed)
  P_inf=1.000  (218s elapsed)
All predictions ready in 237s


## Table column 1 — Uniform Prior

For each sequence, find the α minimizing mean JS distance between predicted
and empirical proportions across the 5 rounds.

In [18]:
# Paper row order: Informative -> Persuade-up -> Persuade-down
PAPER_ROW_ORDER = [
    ("informative", 0), ("informative", 1),
    ("pers_plus", 0), ("pers_plus", 1), ("pers_plus", 2),
    ("pers_minus", 0), ("pers_minus", 1), ("pers_minus", 2),
]

def best_alpha_per_seq(preds_by_seq, empirical_psi):
    """Returns dict {seq: (best_alpha, best_mean_js)}."""
    out = {}
    for seq, alpha_dict in preds_by_seq.items():
        emp = empirical_psi[seq]
        best_js = np.inf
        best_alpha = None
        for alpha, pred in alpha_dict.items():
            js = float(np.mean([jensenshannon(emp[r], pred[r]) for r in range(5)]))
            if js < best_js:
                best_js = js
                best_alpha = float(alpha)
        out[seq] = (best_alpha, best_js)
    return out

results_uniform = best_alpha_per_seq(preds_uniform, empirical_psi)
mean_js_uniform = float(np.mean([js for _, js in results_uniform.values()]))

print(f"Uniform Prior [1/3, 1/3, 1/3] — mean JS across 8 sequences: {mean_js_uniform:.4f}")
print()
print(f"  {'sequence':22s}  {'α':>9}  {'JS':>7}")
for seq in PAPER_ROW_ORDER:
    a, j = results_uniform[seq]
    print(f"  {seq[0]:13s}[{seq[1]}]      {a:9.3f}  {j:7.4f}")


Uniform Prior [1/3, 1/3, 1/3] — mean JS across 8 sequences: 0.2061

  sequence                        α       JS
  informative  [0]         93.293   0.1475
  informative  [1]          1.060   0.1502
  pers_plus    [0]          0.100   0.2577
  pers_plus    [1]          0.100   0.2617
  pers_plus    [2]          0.100   0.2030
  pers_minus   [0]          0.100   0.2244
  pers_minus   [1]          0.100   0.2316
  pers_minus   [2]          0.100   0.1728


/opt/homebrew/Caskroom/miniconda/base/envs/programming-test/lib/python3.13/site-packages/scipy/spatial/distance.py:1388: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(js / 2.0)


## Table column 2 — Truth-Default Prior (sweep over P(inf))

Sweep over candidate priors of the form `[(1−p)/2, p, (1−p)/2]`, picking
the one that minimizes the **mean** JS across all 8 sequences. The paper
reports `[0.25, 0.50, 0.25]` as the recovered prior; we evaluate at the
sweep optimum and additionally at the paper's exact prior so the table
values match.

In [19]:
mean_js_per_prior = {}
results_per_prior = {}
for p_inf, preds_by_seq in predictions.items():
    res = best_alpha_per_seq(preds_by_seq, empirical_psi)
    results_per_prior[p_inf] = res
    mean_js_per_prior[p_inf] = float(np.mean([js for _, js in res.values()]))

best_p_inf_swept = min(mean_js_per_prior, key=mean_js_per_prior.get)
print(f"Sweep best P_inf = {best_p_inf_swept:.3f}  (prior = "
      f"[{(1-best_p_inf_swept)/2:.3f}, {best_p_inf_swept:.3f}, {(1-best_p_inf_swept)/2:.3f}])")
print(f"Sweep best mean JS = {mean_js_per_prior[best_p_inf_swept]:.4f}")

# Print a small table of mean JS over the sweep
print()
print("Mean JS by P_inf:")
for p in sorted(mean_js_per_prior.keys()):
    marker = "  <- best" if p == best_p_inf_swept else ""
    print(f"  P_inf={p:.3f}  mean_JS={mean_js_per_prior[p]:.4f}{marker}")


Sweep best P_inf = 0.500  (prior = [0.250, 0.500, 0.250])
Sweep best mean JS = 0.1671

Mean JS by P_inf:
  P_inf=0.000  mean_JS=0.5067
  P_inf=0.100  mean_JS=0.3642
  P_inf=0.200  mean_JS=0.2829
  P_inf=0.300  mean_JS=0.2198
  P_inf=0.400  mean_JS=0.1851
  P_inf=0.500  mean_JS=0.1671  <- best
  P_inf=0.600  mean_JS=0.1676
  P_inf=0.700  mean_JS=0.1816
  P_inf=0.800  mean_JS=0.2073
  P_inf=0.900  mean_JS=0.2511
  P_inf=1.000  mean_JS=0.4546


In [20]:
# Paper row order: Informative -> Persuade-up -> Persuade-down
PAPER_ROW_ORDER = [
    ("informative", 0), ("informative", 1),
    ("pers_plus", 0), ("pers_plus", 1), ("pers_plus", 2),
    ("pers_minus", 0), ("pers_minus", 1), ("pers_minus", 2),
]

# Evaluate at exactly the paper's prior — the predictions table uses our
# discrete grid, but P_inf=0.50 falls on the grid (0.20 + 12*0.025 = 0.50).
PAPER_P_INF = 0.5
# Find the closest grid point (should be exact)
nearest = min(P_INF_GRID, key=lambda p: abs(p - PAPER_P_INF))
print(f"Using P_inf = {nearest:.3f} ")
results_truth = results_per_prior[nearest]
mean_js_truth = float(np.mean([js for _, js in results_truth.values()]))

print(f"\nTruth-Default Prior [0.25, 0.50, 0.25] — mean JS: {mean_js_truth:.4f}")
print()
print(f"  {'sequence':22s}  {'α':>9}  {'JS':>7}")
for seq in PAPER_ROW_ORDER:
    a, j = results_truth[seq]
    print(f"  {seq[0]:13s}[{seq[1]}]      {a:9.3f}  {j:7.4f}")


Using P_inf = 0.500 

Truth-Default Prior [0.25, 0.50, 0.25] — mean JS: 0.1671

  sequence                        α       JS
  informative  [0]          7.402   0.2005
  informative  [1]          2.049   0.1491
  pers_plus    [0]          0.100   0.1681
  pers_plus    [1]          0.100   0.1972
  pers_plus    [2]          0.264   0.1601
  pers_minus   [0]          0.107   0.1875
  pers_minus   [1]          0.100   0.1616
  pers_minus   [2]          0.238   0.1126


## Final table — both columns side by side

In [21]:
# Paper row order: Informative -> Persuade-up -> Persuade-down
PAPER_ROW_ORDER = [
    ("informative", 0), ("informative", 1),
    ("pers_plus", 0), ("pers_plus", 1), ("pers_plus", 2),
    ("pers_minus", 0), ("pers_minus", 1), ("pers_minus", 2),
]

print(f"{'Sequence':22s} | {'Uniform Prior':24s} | {'Truth-Default Prior':24s}")
print(f"{'':22s} | {'[1/3, 1/3, 1/3]':24s} | {'[0.25, 0.50, 0.25]':24s}")
print(f"{'':22s} | {'α':>9}  {'JS':>10}  | {'α':>9}  {'JS':>10}")
print("-" * 84)
for seq in PAPER_ROW_ORDER:
    a_u, j_u = results_uniform[seq]
    a_t, j_t = results_truth[seq]
    print(f"  {seq[0]:13s}[{seq[1]}]      | {a_u:9.3f}  {j_u:10.4f}  | {a_t:9.3f}  {j_t:10.4f}")
print("-" * 84)
print(f"  {'Mean JS':19s}      | {'':9s}  {mean_js_uniform:10.4f}  | {'':9s}  {mean_js_truth:10.4f}")


Sequence               | Uniform Prior            | Truth-Default Prior     
                       | [1/3, 1/3, 1/3]          | [0.25, 0.50, 0.25]      
                       |         α          JS  |         α          JS
------------------------------------------------------------------------------------
  informative  [0]      |    93.293      0.1475  |     7.402      0.2005
  informative  [1]      |     1.060      0.1502  |     2.049      0.1491
  pers_plus    [0]      |     0.100      0.2577  |     0.100      0.1681
  pers_plus    [1]      |     0.100      0.2617  |     0.100      0.1972
  pers_plus    [2]      |     0.100      0.2030  |     0.264      0.1601
  pers_minus   [0]      |     0.100      0.2244  |     0.107      0.1875
  pers_minus   [1]      |     0.100      0.2316  |     0.100      0.1616
  pers_minus   [2]      |     0.100      0.1728  |     0.238      0.1126
------------------------------------------------------------------------------------
  Mean JS           